In [ ]:
import os
import gc
import torch
import pandas as pd
import numpy as np
import random
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

from datasets import ClassLabel, Features, Value

# ---------------------------------------------------------
# [환경 설정] 메모리 관리 및 시드 고정
# ---------------------------------------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
gc.collect()
torch.cuda.empty_cache()

SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로드 및 전처리 (Text Augmentation 적용)
# ---------------------------------------------------------
DATA_PATH = "./data/munch_labeled_FINAL_190139.csv"
df = pd.read_csv(DATA_PATH)

# 필요한 컬럼 확인 및 결측치 제거
df = df[['input_sentence', 'intensity', 'field', 'score']].dropna()

# [Target Label 생성] Score 4점 이상 = 1 (Hard), 미만 = 0 (Easy)
# 데이터를 정수형(int)으로 명확하게 변환합니다.
df['label'] = df['score'].apply(lambda x: 1 if x >= 4 else 0).astype(int)

# [Feature Engineering] 메타데이터를 텍스트에 주입
def combine_features(row):
    intensity_tag = f"[{str(row['intensity']).upper()}]"
    field_tag = f"[{str(row['field']).upper()}]"
    return f"{intensity_tag} {field_tag} {row['input_sentence']}"

df['text'] = df.apply(combine_features, axis=1)

# 데이터셋 생성
dataset = Dataset.from_pandas(df[['text', 'label']])

# ---------------------------------------------------------
# [FIX] 에러 해결: label 컬럼을 ClassLabel 타입으로 캐스팅
# ---------------------------------------------------------
# stratify_by_column을 사용하려면 해당 컬럼이 ClassLabel 타입이어야 합니다.
dataset = dataset.cast_column("label", ClassLabel(num_classes=2, names=["Easy", "Hard"]))

# 데이터 분할 (이제 에러가 발생하지 않습니다)
train_test_split = dataset.train_test_split(test_size=0.15, seed=SEED, stratify_by_column="label")

dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

print(f"✅ 데이터 준비 완료: Train({len(dataset_dict['train'])}) / Test({len(dataset_dict['test'])})")
print(f"🔍 입력 예시: {dataset_dict['train'][0]['text']}")
print(f"🔍 라벨 타입 확인: {dataset_dict['train'].features['label']}")

# ---------------------------------------------------------
# 2. 클래스 가중치 계산 (불균형 데이터 대응)
# ---------------------------------------------------------
# 학습 데이터의 라벨 분포를 계산하여 Loss 가중치 설정
train_labels = dataset_dict['train']['label']
class_weights = compute_class_weight(
    class_weight="balanced", 
    classes=np.unique(train_labels), 
    y=train_labels
)
# Tensor로 변환 및 GPU 할당
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"⚖️ Class Weights: Easy(0)={class_weights[0]:.2f}, Hard(1)={class_weights[1]:.2f}")

# ---------------------------------------------------------
# 3. 모델 및 토크나이저 초기화
# ---------------------------------------------------------
MODEL_NAME = "klue/roberta-base" # 한국어/영어 혼용에 적합
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2  # Binary Classification
).to(device)

# ---------------------------------------------------------
# 4. Custom Trainer 정의 (Weighted Loss 적용)
# ---------------------------------------------------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # inputs에서 labels 추출
        labels = inputs.get("labels")
        # 모델 순전파
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Weighted CrossEntropyLoss 계산
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

# ---------------------------------------------------------
# 5. 학습 설정
# ---------------------------------------------------------
OUTPUT_DIR = "./output/smart_router_classifier"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,              # Epoch 조절
    learning_rate=2e-5,              # 분류 모델은 조금 더 낮은 LR 권장
    per_device_train_batch_size=16,  # 메모리 안전 설정 유지
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",      # Accuracy보다 F1 Score가 중요 (불균형 고려)
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=0,
    report_to="none",
    seed=SEED
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    
    return {"accuracy": acc, "f1": f1}

# ---------------------------------------------------------
# 6. 학습 및 평가 실행
# ---------------------------------------------------------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🔥 [Classification Mode] 학습 시작!")
trainer.train()

# ---------------------------------------------------------
# 7. 최종 상세 리포트
# ---------------------------------------------------------
print("\n📊 최종 성능 평가:")
predictions = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# 분류 리포트 출력 (Precision, Recall 확인 중요)
print(classification_report(y_true, y_pred, target_names=['Easy(0)', 'Hard(1)']))

trainer.save_model(OUTPUT_DIR)
print(f"💾 모델 저장 완료: {OUTPUT_DIR}")